In [ ]:
import numpy as np, pandas as pd, requests
import csv, gc, glob, io, os, re, shutil, time, zipfile
import pyarrow as pa, pyarrow.compute as pc, pyarrow.csv as pacsv, pyarrow.parquet as pq
from pathlib import Path
from urllib.parse import urljoin, urlparse
from bs4 import BeautifulSoup

BASE_DIR   = Path("") 

RAW_DIR = BASE_DIR / "Data" / "Raw"
RAW_DIR.mkdir(parents=True, exist_ok=True) 

INTERIM_DIR = BASE_DIR / "Data" / "Interim"
INTERIM_DIR.mkdir(parents=True, exist_ok=True) 

PROCESSED_DIR = BASE_DIR / "Data" / "Processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True) 

MERGED_DIR = BASE_DIR / "Data" / "Merged" 
MERGED_DIR.mkdir(parents=True, exist_ok=True) 

FINAL_DIR = BASE_DIR / "Data" / "Final"
FINAL_DIR.mkdir(parents=True, exist_ok=True)

In [3]:
RAW_DIR_PRESC = RAW_DIR / "Prescribing_Data"
RAW_DIR_PRESC.mkdir(parents=True, exist_ok=True) 

RAW_DIR_PRESC_2022 = RAW_DIR_PRESC / "2022"
RAW_DIR_PRESC_2023 = RAW_DIR_PRESC / "2023"
RAW_DIR_PRESC_2024 = RAW_DIR_PRESC / "2024"
RAW_DIR_PRESC_2025 = RAW_DIR_PRESC / "2025"
RAW_DIR_PRESC_2026 = RAW_DIR_PRESC / "2026"

INTERIM_DIR = BASE_DIR / "Data" / "Interim"
INTERIM_DIR_PRESC = INTERIM_DIR / "Prescribing_Data"
INTERIM_DIR_PRESC.mkdir(parents=True, exist_ok=True) 

PROCESSED_DIR = BASE_DIR / "Data" / "Processed"
PROCESSED_DIR_PRESC = PROCESSED_DIR / "Prescribing_Data"

# BNF

In [4]:
bnf = pd.read_csv(PROCESSED_DIR / "bnf_processed.csv", dtype=str)
bnf.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 440 entries, 0 to 439
Data columns (total 16 columns):
 #   Column                       Non-Null Count  Dtype 
---  ------                       --------------  ----- 
 0   Unnamed: 0                   440 non-null    object
 1   YEAR_MONTH                   440 non-null    object
 2   BNF_CHAPTER                  440 non-null    object
 3   BNF_CHAPTER_CODE             440 non-null    object
 4   BNF_SECTION                  440 non-null    object
 5   BNF_SECTION_CODE             440 non-null    object
 6   BNF_PARAGRAPH                440 non-null    object
 7   BNF_PARAGRAPH_CODE           440 non-null    object
 8   BNF_SUBPARAGRAPH             440 non-null    object
 9   BNF_SUBPARAGRAPH_CODE        440 non-null    object
 10  BNF_CHEMICAL_SUBSTANCE       440 non-null    object
 11  BNF_CHEMICAL_SUBSTANCE_CODE  440 non-null    object
 12  BNF_PRODUCT                  440 non-null    object
 13  BNF_PRODUCT_CODE             440 no

In [5]:
# Unique chemical substance code + name pairs
substance_list = (
    bnf[["BNF_CHEMICAL_SUBSTANCE_CODE", "BNF_CHEMICAL_SUBSTANCE"]]
    .drop_duplicates()
    .sort_values("BNF_CHEMICAL_SUBSTANCE_CODE")
    .reset_index(drop=True)
)

# Counts
n_codes = bnf["BNF_CHEMICAL_SUBSTANCE_CODE"].nunique()
n_names = bnf["BNF_CHEMICAL_SUBSTANCE"].nunique()
n_pairs = len(substance_list)

print(f"Unique BNF_CHEMICAL_SUBSTANCE_CODEs: {n_codes}")
print(f"Unique BNF_CHEMICAL_SUBSTANCE names: {n_names}")
print(f"Unique code/name pairs: {n_pairs}")

# Show the list
print(substance_list.to_string(index=False))

# Optional: as a dictionary {code: name}
chem_dict = dict(zip(substance_list["BNF_CHEMICAL_SUBSTANCE_CODE"],
                     substance_list["BNF_CHEMICAL_SUBSTANCE"]))

Unique BNF_CHEMICAL_SUBSTANCE_CODEs: 27
Unique BNF_CHEMICAL_SUBSTANCE names: 27
Unique code/name pairs: 27
BNF_CHEMICAL_SUBSTANCE_CODE                BNF_CHEMICAL_SUBSTANCE
                  0409010A0             Apomorphine hydrochloride
                  0409010AA                            Safinamide
                  0409010AB                             Opicapone
                  0409010AC Apomorphine hydrochloride hemihydrate
                  0409010B0              Amantadine hydrochloride
                  0409010D0                             Carbidopa
                  0409010H0              Ropinirole hydrochloride
                  0409010I0                              Levodopa
                  0409010K0   Co-beneldopa (Benserazide/levodopa)
                  0409010N0     Co-careldopa (Carbidopa/levodopa)
                  0409010S0                             Tolcapone
                  0409010T0              Selegiline hydrochloride
                  0409010U0        

# Download

In [6]:
# Set of codes from the list you just built (set = fast membership checks)
SUBSTANCE_CODES = set(substance_list["BNF_CHEMICAL_SUBSTANCE_CODE"].astype(str).str.strip())
SUBSTANCE_ARRAY = pa.array(sorted(SUBSTANCE_CODES), type=pa.string())

DATASET_ID = "english-prescribing-dataset-epd-with-snomed-code"
API_URL = f"https://opendata.nhsbsa.net/api/3/action/package_show?id={DATASET_ID}"


def get_resources():
    resp = requests.get(API_URL, timeout=60)
    resp.raise_for_status()
    return resp.json()["result"]["resources"]


def download_year(year, start_month=1, end_month=12):
    year = str(year)
    raw_dir = RAW_DIR_PRESC / year
    raw_dir.mkdir(parents=True, exist_ok=True)

    targets = []
    for r in get_resources():
        name = (r.get("name") or "").upper()
        for m in range(start_month, end_month + 1):
            if f"{year}{m:02d}" in name:
                targets.append((f"{year}{m:02d}", r["url"]))
                break
    targets.sort()
    print(f"{len(targets)} months found for {year} (months {start_month}-{end_month})\n")

    for month, url in targets:
        final_path = raw_dir / f"epd_snomed_{month}.parquet"
        part_path = raw_dir / f"epd_snomed_{month}.parquet.part"
        tmp_path = raw_dir / f"download_{month}.tmp"

        if final_path.exists():
            print(f"{month} already done, skipping")
            continue

        t0 = time.time()
        writer = None
        try:
            # 1. Download the full CSV to disk
            with requests.get(url, stream=True, timeout=120) as r:
                r.raise_for_status()
                with open(tmp_path, "wb") as f:
                    for chunk in r.iter_content(chunk_size=1024 * 1024):
                        f.write(chunk)

            # 2. Filter to the chemical substance codes and write parquet
            if part_path.exists():
                part_path.unlink()

            with open(tmp_path, "r", encoding="utf-8-sig", newline="") as f:
                names = next(csv.reader(f))
            schema = pa.schema([(n, pa.string()) for n in names])
            has_chem = "BNF_CHEMICAL_SUBSTANCE_CODE" in names

            reader = pacsv.open_csv(
                tmp_path,
                read_options=pacsv.ReadOptions(block_size=64 * 1024 * 1024),
                parse_options=pacsv.ParseOptions(
                    newlines_in_values=True,
                    invalid_row_handler=lambda row: "skip",
                ),
                convert_options=pacsv.ConvertOptions(
                    column_types={n: pa.string() for n in names}
                ),
            )

            writer = pq.ParquetWriter(part_path, schema, compression="snappy")
            kept = total = 0
            try:
                for batch in reader:
                    tbl = pa.Table.from_batches([batch])
                    total += tbl.num_rows

                    if has_chem:
                        codes = pc.utf8_trim_whitespace(tbl["BNF_CHEMICAL_SUBSTANCE_CODE"])
                    else:
                        codes = pc.utf8_slice_codeunits(
                            pc.utf8_trim_whitespace(tbl["BNF_CODE"]), 0, 9
                        )

                    tbl = tbl.filter(pc.is_in(codes, value_set=SUBSTANCE_ARRAY))
                    kept += tbl.num_rows
                    if tbl.num_rows:
                        writer.write_table(tbl)
            finally:
                reader.close()      # release the file handle so the .tmp can be deleted
                del reader
                gc.collect()

            writer.close()
            writer = None

            # 3. Final name only once fully written
            os.replace(part_path, final_path)
            print(f"Finished {month}: kept {kept:,} of {total:,} rows, "
                  f"{final_path.stat().st_size / 1e6:,.0f} MB, {(time.time() - t0) / 60:.1f} min")

        except Exception as e:
            print(f" {month} failed: {e}")
            if writer is not None:
                try: writer.close()
                except Exception: pass
            if part_path.exists():
                try: part_path.unlink()
                except PermissionError: pass

        finally:
            try:
                if tmp_path.exists():
                    tmp_path.unlink()
            except PermissionError:
                print(f"   (couldn't delete {tmp_path.name}, remove it manually)")

In [7]:
download_year(2022, start_month=7)
download_year(2023)
download_year(2024)
download_year(2025)
download_year(2026, end_month=3)

6 months found for 2022 (months 7-12)

202207 already done, skipping
202208 already done, skipping
202209 already done, skipping
202210 already done, skipping
202211 already done, skipping
202212 already done, skipping
12 months found for 2023 (months 1-12)

202301 already done, skipping
202302 already done, skipping
202303 already done, skipping
202304 already done, skipping
202305 already done, skipping
202306 already done, skipping
202307 already done, skipping
202308 already done, skipping
202309 already done, skipping
202310 already done, skipping
202311 already done, skipping
202312 already done, skipping
12 months found for 2024 (months 1-12)

202401 already done, skipping
202402 already done, skipping
202403 already done, skipping
202404 already done, skipping
202405 already done, skipping
202406 already done, skipping
202407 already done, skipping
202408 already done, skipping
202409 already done, skipping
202410 already done, skipping
202411 already done, skipping
202412 alre

# Merge all years

In [8]:
YEARS = [2022, 2023, 2024, 2025, 2026]

CODE, NAME = "BNF_CHEMICAL_SUBSTANCE_CODE", "BNF_CHEMICAL_SUBSTANCE"
KEYS = ["YEAR_MONTH", "PRACTICE_CODE", "ICB_CODE", "BNF_CHAPTER_PLUS_CODE", CODE]
NUMS = ["ITEMS", "NIC", "ACTUAL_COST"]
NAME_ALTS = [NAME, "CHEMICAL_SUBSTANCE_BNF_DESCR"]          # tried in this order
out_path = INTERIM_DIR_PRESC / "panel_all_years.parquet"
tmp_path = INTERIM_DIR_PRESC / "panel_all_years.parquet.part"

# 1. Build the panel from the raw monthly files
def load_month(f):
    names = pq.ParquetFile(f).schema.names
    name_col = next((c for c in NAME_ALTS if c in names), None)
    need = ["YEAR_MONTH", "PRACTICE_CODE", "ICB_CODE", "BNF_CHAPTER_PLUS_CODE"] + NUMS
    need += [CODE] if CODE in names else ["BNF_CODE"]
    if name_col:
        need.append(name_col)
    d = pd.read_parquet(f, columns=list(dict.fromkeys(need)))     # only the columns we need

    if CODE not in d:                                             # older files: first 9 chars of BNF_CODE
        d[CODE] = d["BNF_CODE"].str.strip().str[:9]
        d = d.drop(columns="BNF_CODE")
    d[CODE] = d[CODE].str.strip()
    d["PRACTICE_CODE"] = d["PRACTICE_CODE"].str.strip()

    # YEAR_MONTH: raw is 202201, panel/DROP_MONTHS use 2022-01
    ym = d["YEAR_MONTH"].str.strip()
    d["YEAR_MONTH"] = ym.where(ym.str.contains("-"), ym.str[:4] + "-" + ym.str[4:6])

    for c in NUMS:
        d[c] = pd.to_numeric(d[c], errors="coerce")

    # names kept separately so they can't split rows during aggregation
    names_df = None
    if name_col:
        names_df = d[["YEAR_MONTH", CODE, name_col]].rename(columns={name_col: NAME}).drop_duplicates()
        d = d.drop(columns=name_col)

    agg = d.groupby(KEYS, dropna=False, as_index=False)[NUMS].sum()
    return agg, names_df

parts, name_parts, n_raw = [], [], 0
for y in YEARS:
    files = sorted((RAW_DIR_PRESC / str(y)).glob("epd_snomed_*.parquet"))
    print(f"{y}: {len(files)} monthly files")
    for f in files:
        agg, nm = load_month(f)
        parts.append(agg)
        if nm is not None:
            name_parts.append(nm)
        n_raw += 1

panel = pd.concat(parts, ignore_index=True)
print(f"Loaded {len(panel):,} rows from {n_raw} monthly files")

# 2. One name per substance code: the name from its latest month (prefer non-"(old)")
if name_parts:
    pairs = pd.concat(name_parts, ignore_index=True).drop_duplicates()
    pairs = pairs[pairs["YEAR_MONTH"].isin(set(panel["YEAR_MONTH"]))]
else:                                                             # no name column in the raw files
    pairs = bnf[[CODE, NAME]].drop_duplicates().assign(YEAR_MONTH="")   # fall back to bnf reference
pairs["is_old"] = pairs[NAME].str.contains(r"\(old\)", case=False, na=False)
pairs = pairs.sort_values([CODE, "YEAR_MONTH", "is_old"], ascending=[True, True, False])
lookup = pairs.groupby(CODE).tail(1).set_index(CODE)[NAME].to_dict()
n_multi = pairs.groupby(CODE)[NAME].nunique().gt(1).sum()
print(f"{len(lookup):,} substance codes, {n_multi} had more than one name")

before = panel[NUMS].sum()
panel[NAME] = panel[CODE].map(lookup)

# 3. Save (write to .part first, then rename)
if tmp_path.exists():
    tmp_path.unlink()
panel.to_parquet(tmp_path, index=False, compression="snappy")
tmp_path.replace(out_path)

# 4. Verify what was saved
chk = pd.read_parquet(out_path)
print(f"\nRows: {len(chk):,} (expected {len(panel):,})")
months = sorted(chk["YEAR_MONTH"].unique())
print(f"Months: {len(months)} ({months[0]} to {months[-1]})")
for c in NUMS:
    print(f"{c}: file {chk[c].sum():,.2f} vs inputs {before[c]:,.2f}")
print(f"Nulls in name: {chk[NAME].isna().sum():,}")
print(f"Nulls in numeric columns: {chk[NUMS].isna().sum().to_dict()}")
dups = chk.duplicated(KEYS).sum()
print(f"Duplicate keys: {dups:,}")

icb_by_year = chk.groupby(chk["YEAR_MONTH"].str[:4])["ICB_CODE"].agg(lambda s: set(s.dropna()))
all_icb = set().union(*icb_by_year)
for y, s in icb_by_year.items():
    if s != all_icb:
        print(f"{y}: ICB codes differ: {sorted(all_icb - s)}")
print(f"Saved to {out_path} ({out_path.stat().st_size / 1e6:,.0f} MB)")

# 5. Check codes and names against bnf
ref = bnf[[CODE, NAME]].drop_duplicates().copy()
got = chk[[CODE, NAME]].drop_duplicates().copy()
for d in (ref, got):
    d[CODE] = d[CODE].astype(str).str.strip()
    d[NAME] = d[NAME].astype(str).str.strip()

ref_codes, got_codes = set(ref[CODE]), set(got[CODE])

# 5a. Code coverage
print(f"\nCodes in bnf:   {len(ref_codes):,}")
print(f"Codes in panel: {len(got_codes):,}")
print(f"In panel but not in bnf: {sorted(got_codes - ref_codes) or 'none'}")
missing = sorted(ref_codes - got_codes)
print(f"In bnf but not in panel: {len(missing)}")
if missing:
    print(f"  (these may simply have no prescribing in the {len(months)} months)")
    print(ref[ref[CODE].isin(missing)].sort_values(CODE).to_string(index=False))

# 5b. Does each code map to exactly one name in each source?
for label, d in [("bnf", ref), ("panel", got)]:
    multi = d.groupby(CODE)[NAME].nunique()
    multi = multi[multi > 1]
    print(f"\n{label}: codes with more than one name: {len(multi)}")
    if len(multi):
        print(d[d[CODE].isin(multi.index)].sort_values(CODE).to_string(index=False))

# 5c. Do the names agree for codes in both?
m = ref.drop_duplicates(CODE).merge(got.drop_duplicates(CODE), on=CODE,
                                    suffixes=("_bnf", "_panel"))
diff = m[m[f"{NAME}_bnf"] != m[f"{NAME}_panel"]]
print(f"\nCodes in both: {len(m):,} | name mismatches: {len(diff)}")
if len(diff):
    print(diff.sort_values(CODE).to_string(index=False))
else:
    print("All names match")

2022: 6 monthly files
2023: 12 monthly files
2024: 12 monthly files
2025: 12 monthly files
2026: 3 monthly files
Loaded 3,291,714 rows from 45 monthly files
27 substance codes, 24 had more than one name

Rows: 3,291,714 (expected 3,291,714)
Months: 45 (2022-07 to 2026-03)
ITEMS: file 37,227,743.00 vs inputs 37,227,743.00
NIC: file 481,035,732.57 vs inputs 481,035,732.57
ACTUAL_COST: file 453,302,233.79 vs inputs 453,302,233.79
Nulls in name: 0
Nulls in numeric columns: {'ITEMS': 0, 'NIC': 0, 'ACTUAL_COST': 0}
Duplicate keys: 0
Saved to /Users/trinityrose/Desktop/TEST/Data/Interim/Prescribing_Data/panel_all_years.parquet (39 MB)

Codes in bnf:   27
Codes in panel: 27
In panel but not in bnf: none
In bnf but not in panel: 0

bnf: codes with more than one name: 0

panel: codes with more than one name: 0

Codes in both: 27 | name mismatches: 2
BNF_CHEMICAL_SUBSTANCE_CODE BNF_CHEMICAL_SUBSTANCE_bnf BNF_CHEMICAL_SUBSTANCE_panel
                  0409010D0                  Carbidopa          

# Processing

In [9]:
# 1. Load the merged panel and slim to the columns needed
pres_slim = pd.read_parquet(
    INTERIM_DIR_PRESC / "panel_all_years.parquet",
    columns=[
        "YEAR_MONTH",
        "PRACTICE_CODE",
        "BNF_CHEMICAL_SUBSTANCE_CODE",
        "BNF_CHEMICAL_SUBSTANCE",
        "ITEMS",
        "NIC",
        "ACTUAL_COST",
    ],
)
display(pres_slim.head())

# 1b. Replace names with the ones in bnf, matched on the substance code
CODE, NAME = "BNF_CHEMICAL_SUBSTANCE_CODE", "BNF_CHEMICAL_SUBSTANCE"
ref = bnf[[CODE, NAME]].copy()
ref[CODE] = ref[CODE].astype(str).str.strip()
ref = ref.drop_duplicates()

dup = ref.groupby(CODE)[NAME].nunique()
print(f"bnf codes with more than one name: {(dup > 1).sum()}")       # should be 0
name_map = ref.drop_duplicates(CODE).set_index(CODE)[NAME]

pres_slim[CODE] = pres_slim[CODE].astype(str).str.strip()
old_names = pres_slim[NAME].copy()
pres_slim[NAME] = pres_slim[CODE].map(name_map).fillna(old_names)    # keep old name if code not in bnf

unmatched = sorted(set(pres_slim.loc[~pres_slim[CODE].isin(name_map.index), CODE]))
print(f"Codes not found in bnf (names left unchanged): {unmatched or 'none'}")
print(f"Rows where the name changed: {(pres_slim[NAME] != old_names).sum():,}")

# 2. Convert YEAR_MONTH (YYYY-MM) to a proper date
pres_slim["YEAR_MONTH_DATE"] = pd.to_datetime(pres_slim["YEAR_MONTH"].astype(str), format="%Y-%m")

# 3. Filter to the study period
pres_slim = pres_slim[
    (pres_slim["YEAR_MONTH_DATE"] >= "2022-07-01") &
    (pres_slim["YEAR_MONTH_DATE"] <= "2026-03-01")
].copy()

# 4. Sum rows that are now duplicates (ICB_CODE and BNF_CHAPTER_PLUS_CODE were dropped)
before = pres_slim[["ITEMS", "NIC", "ACTUAL_COST"]].sum()
n_before = len(pres_slim)
pres_slim = (
    pres_slim
    .groupby(["YEAR_MONTH", "YEAR_MONTH_DATE", "PRACTICE_CODE",
              "BNF_CHEMICAL_SUBSTANCE_CODE", "BNF_CHEMICAL_SUBSTANCE"], as_index=False)
    [["ITEMS", "NIC", "ACTUAL_COST"]].sum()
)

# 5. Checks
print(f"Rows: {n_before:,} -> {len(pres_slim):,} after summing duplicates")
print(f"Months: {pres_slim['YEAR_MONTH_DATE'].nunique()} "
      f"({pres_slim['YEAR_MONTH_DATE'].min():%Y-%m} to {pres_slim['YEAR_MONTH_DATE'].max():%Y-%m})")
for c in ["ITEMS", "NIC", "ACTUAL_COST"]:
    print(f"{c}: {pres_slim[c].sum():,.2f} vs before {before[c]:,.2f}")
print(f"Names per code (should all be 1): {pres_slim.groupby(CODE)[NAME].nunique().max()}")
display(pres_slim.head())

,YEAR_MONTH,PRACTICE_CODE,BNF_CHEMICAL_SUBSTANCE_CODE,BNF_CHEMICAL_SUBSTANCE,ITEMS,NIC,ACTUAL_COST
0,2022-07,-,0409010H0,Ropinirole hydrochloride,3,110.53,103.58687
1,2022-07,-,0409010K0,Co-beneldopa (Benserazide/levodopa),13,96.19,91.30244
2,2022-07,-,0409010N0,Co-careldopa (Carbidopa/levodopa),9,105.08,98.96531
3,2022-07,-,0409010T0,Selegiline hydrochloride,1,9.02,8.54644
4,2022-07,-,0409010V0,Entacapone,1,5.84,5.57302


bnf codes with more than one name: 0
Codes not found in bnf (names left unchanged): none
Rows where the name changed: 2
Rows: 3,291,714 -> 3,291,714 after summing duplicates
Months: 45 (2022-07 to 2026-03)
ITEMS: 37,227,743.00 vs before 37,227,743.00
NIC: 481,035,732.57 vs before 481,035,732.57
ACTUAL_COST: 453,302,233.79 vs before 453,302,233.79
Names per code (should all be 1): 1


,YEAR_MONTH,YEAR_MONTH_DATE,PRACTICE_CODE,BNF_CHEMICAL_SUBSTANCE_CODE,BNF_CHEMICAL_SUBSTANCE,ITEMS,NIC,ACTUAL_COST
0,2022-07,2022-07-01,-,0409010H0,Ropinirole hydrochloride,3,110.53,103.58687
1,2022-07,2022-07-01,-,0409010K0,Co-beneldopa (Benserazide/levodopa),13,96.19,91.30244
2,2022-07,2022-07-01,-,0409010N0,Co-careldopa (Carbidopa/levodopa),9,105.08,98.96531
3,2022-07,2022-07-01,-,0409010T0,Selegiline hydrochloride,1,9.02,8.54644
4,2022-07,2022-07-01,-,0409010V0,Entacapone,1,5.84,5.57302


## Save to processed

In [10]:
out_file = PROCESSED_DIR / "presc_processed.parquet"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

pres_slim.to_parquet(out_file, index=False)

# Quick check by reading it back
chk = pd.read_parquet(out_file)
print(f"Saved {len(chk):,} rows, {chk.shape[1]} columns to {out_file} ({out_file.stat().st_size / 1e6:,.1f} MB)")
print(chk.dtypes)

Saved 3,291,714 rows, 8 columns to /Users/trinityrose/Desktop/TEST/Data/Processed/presc_processed.parquet (39.0 MB)
YEAR_MONTH                             object
YEAR_MONTH_DATE                datetime64[ns]
PRACTICE_CODE                          object
BNF_CHEMICAL_SUBSTANCE_CODE            object
BNF_CHEMICAL_SUBSTANCE                 object
ITEMS                                   int64
NIC                                   float64
ACTUAL_COST                           float64
dtype: object
